In [8]:
import os
import nibabel as nib
import numpy as np
import pandas as pd

In [ ]:
PROJECT_DIR = '/nasdata/archive/personal_project_archives/bule/berner_daten_analyse'

In [2]:
def get_lesion_volume(fname):
    img = nib.load(fname)
    data = img.get_fdata()

    # Get voxel dimensions in mm (from header)
    voxel_sizes = img.header.get_zooms()[:3]  # (x_size, y_size, z_size)
    voxel_volume = np.prod(voxel_sizes)       # mm^3 per voxel

    # Count lesion voxels 
    n_voxels = np.count_nonzero(data)

    # Compute volume in mm^3 and ml
    lesion_volume_mm3 = n_voxels * voxel_volume
    lesion_volume_ml = lesion_volume_mm3 / 1000.0

    return lesion_volume_mm3, lesion_volume_ml

# should be "large"

In [4]:
fname = os.path.join(PROJECT_DIR, 'data', 'patient_results', '76813', 'lesion_msk.nii.gz')
lesion_volume_mm3, lesion_volume_ml = get_lesion_volume(fname)
print(f"Lesion volume: {lesion_volume_mm3:.2f} mm³ ({lesion_volume_ml:.2f} mL)")

Lesion volume: 35335.55 mm³ (35.34 mL)


## should be "small"

In [5]:
fname = os.path.join(PROJECT_DIR, 'data', 'patient_results', '442968', 'lesion_msk.nii.gz')
lesion_volume_mm3, lesion_volume_ml = get_lesion_volume(fname)
print(f"Lesion volume: {lesion_volume_mm3:.2f} mm³ ({lesion_volume_ml:.2f} mL)")

Lesion volume: 3311.27 mm³ (3.31 mL)


## systematic lesion segmentation

In [6]:
segmentation_mask_path= os.path.join(PROJECT_DIR,"data","patient_results")

In [ ]:
not_exists_pth=[]
lesion_volume_df={"Patient_ID":[],"Lesion_volume_mm3":[],"Lesion_volume_ml":[]}
for path in [os.path.join(segmentation_mask_path,path,"lesion_msk.nii.gz") for path in os.listdir(segmentation_mask_path)]:
    
    if not os.path.exists(path):
        not_exists_pth.append(path)
    else:
        lesion_volume_mm3, lesion_volume_ml = get_lesion_volume(path)
        # print(f"{path.split('/')[-2]}: Lesion volume: {lesion_volume_mm3:.2f} mm³ ({lesion_volume_ml:.2f} mL)")
    lesion_volume_df["Patient_ID"].append(path.split('/')[-2])
    lesion_volume_df["Lesion_volume_mm3"].append(lesion_volume_mm3)
    lesion_volume_df["Lesion_volume_ml"].append(lesion_volume_ml)
lesion_volume_df=pd.DataFrame(lesion_volume_df)

lesion_volume_df.to_csv(os.path.join(PROJECT_DIR,"data","clinical","lesion_volume_bern.csv"),index=False)